# 01 — Dataset intake and structural validation

**Purpose:** establish an auditable baseline for the original CSV before changing, cleaning, or modeling it.

**Important:** this notebook does not overwrite the source file. Record the original filename and preserve an untouched copy.

## 1. Load the source dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset(1).csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 2. Schema, types, and missingness

In [ ]:
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.astype(str).rename("dtype").to_frame())

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(3),
    "n_unique": df.nunique(dropna=False)
})
display(profile.sort_values(["missing_count", "n_unique"], ascending=[False, True]))

## 3. Duplicates, constants, and suspicious representations

In [ ]:
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Constant columns:", [c for c in df.columns if df[c].nunique(dropna=False) <= 1])

for c in df.select_dtypes(include=["object"]).columns:
    print(f"\\n{c}: {df[c].nunique(dropna=False)} unique values")
    display(df[c].value_counts(dropna=False).head(20).to_frame("count"))

## 4. Timestamp and derived-time consistency

In [ ]:
if "timestamp" in df.columns:
    ts = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    print("Unparseable timestamps:", int(ts.isna().sum()))
    print("Timestamp min/max:", ts.min(), ts.max())
    if "hour_of_day" in df.columns:
        mismatch = ts.notna() & (df["hour_of_day"].astype("Int64") != ts.dt.hour.astype("Int64"))
        print("hour_of_day mismatches:", int(mismatch.sum()))

## 5. Numeric distributions and category summaries

In [ ]:
display(df.describe(include="all").T)

## 6. Export a compact audit report

In [ ]:
report_path = Path("01_dataset_intake_profile.csv")
profile.to_csv(report_path)
print("Saved:", report_path.resolve())

## Interpretation checklist

- Confirm that row count and columns match the documented source version.
- Explain every missing value, duplicate, constant, and unusual data type.
- Check whether derived fields can be reconstructed from raw fields.
- Do not drop columns or rows in this notebook; document decisions first.